# EDA & Cleaning: eICU Collaborative Research Database Demo

Source: `eicu_crd_demo/2.0.1/`. Real (small, de-identified demo) ICU patient-stays. No genotype data, but has warfarin medication orders, INR/PT lab results (in `lab.xlsx`), and past-history records.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", 100)
DATA_DIR = Path("..").resolve()
DECISIONS_PATH = Path("dataset_decisions.json")

def record_decision(name, verdict, reason, n_rows=None, n_cols=None, saved_to=None):
    decisions = {}
    if DECISIONS_PATH.exists():
        decisions = json.loads(DECISIONS_PATH.read_text())
    decisions[name] = {
        "verdict": verdict,
        "reason": reason,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "saved_to": saved_to,
    }
    DECISIONS_PATH.write_text(json.dumps(decisions, indent=2))
    print(f"Recorded decision for '{name}': {verdict}")


## File inventory: `eicu_crd_demo/2.0.1/` (30 data tables plus a `sqlite/` dump and license/checksum/index files)

**Used:** `medication.xlsx`, `admissiondrug.xlsx` (two independent sources of warfarin orders, a stay counts if it appears in either), `lab.xlsx` (INR/PT results), `pastHistory.xlsx` (comorbidity/history features), and `patient.xlsx` (age, gender, ethnicity, and weight, saved as a separate demographics table).

**Not used:** `admissionDx.xlsx`, `allergy.xlsx`, `apacheApsVar.xlsx`/`apachePatientResult.xlsx`/`apachePredVar.xlsx` (ICU severity-of-illness scores, not warfarin-relevant), `carePlanCareProvider.xlsx`/`carePlanEOL.xlsx`/`carePlanGeneral.xlsx`/`carePlanGoal.xlsx`/`carePlanInfectiousDisease.xlsx` (free-text care-plan notes), `customLab.xlsx` (no INR/PT rows), `diagnosis.xlsx` (ICD diagnosis codes, not joined, could supplement `pastHistory.xlsx`), `hospital.xlsx` (site metadata only), `infusiondrug.xlsx` (IV infusions, not oral warfarin), `intakeOutput.xlsx`, `microLab.xlsx` (microbiology, unrelated to coagulation), `note.xlsx` (free text), `nurseAssessment.xlsx`/`nurseCare.xlsx`/`nurseCharting.xlsx` (nursing flowsheets), `physicalExam.xlsx`, `respiratoryCare.xlsx`/`respiratoryCharting.xlsx`, `treatment.xlsx`, `vitalAperiodic.xlsx`/`vitalPeriodic.xlsx` (vitals), and `sqlite/eicu_v2_0_1.sqlite3.gz` (same data as a raw database dump, redundant with the xlsx exports already read).

In [ ]:
base = DATA_DIR / "usable/eicu_crd_demo/2.0.1"
med = pd.read_excel(base / "medication.xlsx")
admdrug = pd.read_excel(base / "admissiondrug.xlsx")
lab = pd.read_excel(base / "lab.xlsx")
past_hx = pd.read_excel(base / "pastHistory.xlsx")
patient = pd.read_excel(base / "patient.xlsx")
print("medication:", med.shape)
print("admissiondrug:", admdrug.shape)
print("lab:", lab.shape)
print("pastHistory:", past_hx.shape)
print("patient:", patient.shape)

## Warfarin medication signal

In [ ]:
warf_med = med[med["drugname"].astype(str).str.contains("warfarin", case=False, na=False)]
warf_admdrug = admdrug[admdrug["drugname"].astype(str).str.contains("warfarin|coumadin", case=False, na=False)]
print("warfarin rows in medication:", len(warf_med), "| distinct stays:", warf_med["patientunitstayid"].nunique())
print("warfarin rows in admissiondrug:", len(warf_admdrug), "| distinct stays:", warf_admdrug["patientunitstayid"].nunique())

## INR/PT lab signal (found in `lab.xlsx`, not `customLab.xlsx`)

**Correction:** `lab.xlsx`'s `labname` field has two distinct entries here -- `PT` (Prothrombin Time, measured in **seconds**, typically ~10-20s) and `PT - INR` (the actual INR ratio, ~0.8-4). An earlier pass matched both with `labname == "PT"` OR `"inr" in labname`, silently merging two different units into one series -- which inflated the apparent IQR-outlier rate in notebook 12 (looked like 21.5% vs. ~4-8% for MIMIC). Fixed below to use `PT - INR` only.

In [ ]:
print(sorted(lab["labname"].unique())[:5], "... (checking for PT/INR specifically)")
inr_names = ["PT - INR"]  # exclude plain "PT" -- that's prothrombin time in seconds, not the INR ratio
print("Matched lab names:", inr_names)
inr = lab[lab["labname"].isin(inr_names)]
print("INR lab rows:", inr.shape, "| distinct stays:", inr["patientunitstayid"].nunique())
inr.head(5)

## Overlap: patients with BOTH a warfarin order and an INR/PT measurement

This is the actual usable cohort. A stay with warfarin but no INR (or vice versa) can not support dose/INR prediction.

In [ ]:
warf_stays = set(warf_med["patientunitstayid"]) | set(warf_admdrug["patientunitstayid"])
inr_stays = set(inr["patientunitstayid"])
usable_stays = warf_stays & inr_stays
print(f"Distinct stays with warfarin order: {len(warf_stays)}")
print(f"Distinct stays with INR/PT lab: {len(inr_stays)}")
print(f"Overlap (usable for dose/INR analysis): {len(usable_stays)}")

## Missingness on the INR lab result itself

In [ ]:
print((inr[inr["patientunitstayid"].isin(usable_stays)]["labresult"].isna().mean() * 100).round(1), "% missing labresult")
inr[inr["patientunitstayid"].isin(usable_stays)]["labresult"].describe()

## Cleaning
Keep warfarin medication/admission-drug rows and INR/PT lab rows restricted to the 160 overlapping stays, plus their past-history rows (admission/comorbidity history, the closest eICU equivalent to MIMIC's diagnosis history). No genotype in this cohort, same limitation as MIMIC-III/IV.

In [ ]:
warf_med_clean = warf_med[warf_med["patientunitstayid"].isin(usable_stays)]
warf_admdrug_clean = warf_admdrug[warf_admdrug["patientunitstayid"].isin(usable_stays)]
inr_clean = inr[inr["patientunitstayid"].isin(usable_stays)]
hx_clean = past_hx[past_hx["patientunitstayid"].isin(usable_stays)]

# Demographics from patient.xlsx. eICU encodes age over 89 as the string "> 89"; map that to 90.
demo_clean = patient[patient["patientunitstayid"].isin(usable_stays)][
    ["patientunitstayid", "age", "gender", "ethnicity", "admissionweight", "admissionheight"]
].copy()
demo_clean["age"] = demo_clean["age"].replace("> 89", "90")
demo_clean["age_years"] = pd.to_numeric(demo_clean["age"], errors="coerce")
demo_clean = demo_clean.drop(columns=["age"])

warf_med_clean.to_csv("cleaned/eicu_warfarin_medication_clean.csv", index=False)
warf_admdrug_clean.to_csv("cleaned/eicu_warfarin_admissiondrug_clean.csv", index=False)
inr_clean.to_csv("cleaned/eicu_inr_labs_clean.csv", index=False)
hx_clean.to_csv("cleaned/eicu_pasthistory_clean.csv", index=False)
demo_clean.to_csv("cleaned/eicu_patient_demographics_clean.csv", index=False)
print(f"Saved cleaned tables for {len(usable_stays)} warfarin+INR stays")
print("age_years summary:\n", demo_clean["age_years"].describe())

## Verdict

**KEEP: real-world clinical-history validation source (no genotype).** 160 patient-stays have both a warfarin order and an INR/PT measurement, more than MIMIC-III (17) and MIMIC-IV (23) combined. Same role as those two cohorts: clinical-only external validation, not genotype-aware training.

In [ ]:
record_decision(
    "eicu_crd_demo",
    verdict="KEEP",
    reason="Has 160 real patient-stays with both a warfarin order and an INR/PT lab result; no genotype. Use for clinical-only validation alongside MIMIC-III/IV.",
    n_rows=int(len(usable_stays)), n_cols=None, saved_to="cleaned/eicu_*_clean.csv",
)